# 01 the contact and booking mail contract

What a visitor's enquiry needs in order to arrive, checked against the site
that is actually deployed rather than against this checkout.

On 2026-09-07 at 18:40 UTC an audience script ran `templateUpdate` over every
row of this tenant with `"envelope": {"subject": ..., "to": []}`, and
emailwerk writes `to: input.to ?? []`, so all four request templates of this
brand lost their recipients. The site sends anonymously and the anonymous
branch takes the recipients from the template's stored envelope, so from that
moment every contact and every booking on limosen.at answered
`PUBLIC_SEND_NO_STORED_RECIPIENT` and the visitor read "Something went wrong."
It stayed that way for ten days and nine deploys, because nothing looked at the
live templates after an upload. Section 6 below is that look.

The notebook is read only. It fetches the served page and its bundle, asks
emailwerk and this brand's pylon the questions a browser would ask, and reads
the templates with the admin credential when there is one. Nothing is sent:
the one probe that reaches the send resolver carries a deliberately invalid
reply-to, which is refused during argument validation, before the template is
even looked up. A real send is a recipe in the last markdown cell, for a human,
because it puts mail in the company's inbox.

Configuration, all optional, all with this brand's own defaults:
`SITE_URL` (what to fetch, a `*.pages.dev` deployment when the workflow is
asking whether a rollback would help), `SITE_ORIGIN` (the origin a browser
would send, which stays `https://limosen.at`), `EMAILWERK_AUTH` as `user:pass`
for the admin half, `SMOKE_DRILL` for the drill, and `CI`, which turns a
missing credential from a skip into a failure.

In [ ]:
import json
import os
import pathlib
import re
import sys

# The notebook runs from tests/ under nbconvert and from the repository root by
# hand, so both are offered to the import.
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "tests"))
import sitekit as k

k.start_run("01-contact-mail-contract")

# The recipients, the sender and the languages are not retyped here. They belong
# to scripts/create-mail-templates.py, which is what writes them to emailwerk,
# so a drift between the script and the live rows shows up as a failure instead
# of the test quietly agreeing with itself.
CREATE = k.load_create_script()
RECIPIENTS = set(CREATE.RECIPIENTS) if CREATE else set()
SENDER_ID = CREATE.SENDER_ID if CREATE else ""
LANGUAGES = list(CREATE.LANGUAGES) if CREATE else ["de", "en", "tr", "ar"]

IDS = k.repo_ids()
REPO_REQUEST = k.repo_request_ids(LANGUAGES)

print("languages:", LANGUAGES)
print("recipients from the script:", sorted(RECIPIENTS))
print("ids.json entries:", len(IDS))

## 1. The site under test serves a page and a bundle

Everything below reads the deployed bundle, so a missing or truncated bundle
has to be a failure of its own rather than four confusing failures further
down. The bundle also has to carry the mail gateway's URL: the whole contract
is about a call the browser makes to `emailwerk.com`, and a build that no
longer contains that string is a build that cannot make it.

In [ ]:
SITE = k.fetch_site()

with k.check("the site serves a page, a bundle and the mail gateway's URL") as c:
    page = c.require(SITE["page"])
    c.expect(page.status == 200, f"GET {k.CONFIG['site_url']}/ answers 200 (got {page.status})")
    c.expect(bool(SITE["bundle_url"]), "the HTML loads an /app-<hash>.js bundle")
    if SITE["bundle_url"]:
        c.note("bundle: " + SITE["bundle_url"])
        asset = c.require(SITE["asset"])
        c.expect(asset.status == 200, f"the bundle is served (got {asset.status})")
        c.expect(len(SITE["bundle"]) > 100_000,
                 f"the bundle is a whole bundle, {len(SITE['bundle'])} bytes")
        c.expect(k.CONFIG["emailwerk_url"] in SITE["bundle"],
                 f"the bundle names {k.CONFIG['emailwerk_url']}")

## 2. The locale map the deployed bundle carries

`src/services/mail-templates.ts` is inlined into the bundle, so the object it
exports survives minification as `de:"c…",en:"c…",tr:"c…",ar:"c…"`. That object,
not `mail-templates/ids.json`, is the answer to "which templates does the
deployed site send", and every check further down uses it.

A difference between the two is not a failure. It means the live site was built
from a different commit than this checkout, which is ordinary between a
template run and the next deploy, so it is recorded as a warning that names
both sides.

In [ ]:
DEPLOYED = SITE["templates"]

with k.check("the deployed bundle carries a locale to template map") as c:
    c.expect(bool(DEPLOYED), "the de/en/tr/ar object is in the bundle")
    for lang in LANGUAGES:
        c.expect(bool(DEPLOYED.get(lang)), f"{lang} has a template id")
    c.note("deployed: " + json.dumps(DEPLOYED))

with k.check("the deployed ids are the ids this checkout would send") as c:
    if not DEPLOYED:
        c.fail("no map in the bundle, nothing to compare")
    elif not REPO_REQUEST:
        c.skip("mail-templates/ids.json was not found")
    else:
        drift = {lang: (DEPLOYED.get(lang), REPO_REQUEST.get(lang))
                 for lang in LANGUAGES if DEPLOYED.get(lang) != REPO_REQUEST.get(lang)}
        if drift:
            c.warn("the deployed site was not built from this checkout: "
                   + ", ".join(f"{lang} deployed {a} / ids.json {b}"
                               for lang, (a, b) in sorted(drift.items())))
        else:
            c.note("deployed and ids.json agree on all four languages")

## 3. Nothing of the sibling brand is in what the visitor downloads

`scripts/deploy.sh` refuses a build that carries KRC's address, phone number or
name, and it caught a real case on its first run. This is the same question
asked of the site that is live, which is the only version a visitor ever sees,
and it is asked of the HTML as well as of the bundle, because the page metadata
is rendered into the HTML and never reaches the JavaScript.
A third check reads this repository's own `.github/workflows/`. Nothing else in
the suite does: the two mechanisms the first hard rule names both look at what
the build produced, and a workflow that names the sibling's Cloudflare Pages
project would publish this tree onto the other company's domain without one
string of theirs ever appearing in a build here.

In [ ]:
with k.check("neither the HTML nor the bundle names the other company") as c:
    for where, text in (("the HTML of /", SITE["html"]), ("the bundle", SITE["bundle"])):
        if not text:
            c.fail(f"{where} is empty, nothing was checked")
            continue
        found = [marker for marker in k.foreign_markers() if marker in text]
        c.expect(not found, f"{where} is clean of {k.foreign_markers()} (found {found})")

with k.check("this company's own phone number is in the page a visitor reads") as c:
    c.expect("660 876 06 06" in SITE["html"],
             "the HTML of / carries LIMOSEN's number, so the build is this brand's")

# The two mechanisms the first hard rule names, the deploy script's grep and the
# checks above, both read what the build produced. Neither of them ever looks at
# `.github/`, and a workflow that names the sibling's Pages project would put
# this tree on the other company's domain without a single string of theirs
# appearing in the build. That is not hypothetical: the sibling repository's own
# deploy workflow named this project for months.
with k.check("no workflow of this repository deploys the sibling's Pages project") as c:
    root = k.repo_root()
    workflows = sorted((root / ".github" / "workflows").glob("*.y*ml")) if root else []
    if not workflows:
        c.skip("no .github/workflows to read, this is not a full checkout")
    else:
        named = re.compile(r"(?:CLOUDFLARE_PROJECT_NAME|CF_PAGES_PROJECT)\s*:\s*['\"]?([A-Za-z0-9_-]+)")
        for path in workflows:
            for project in named.findall(path.read_text(encoding="utf-8")):
                c.expect(project == k.CF_PAGES_PROJECT,
                         f"{path.name} deploys {project!r}, this brand's project "
                         f"is {k.CF_PAGES_PROJECT!r}")
        c.note("read: " + ", ".join(p.name for p in workflows))

## 4. The browser is allowed to call emailwerk at all

The form posts from the visitor's browser to another origin, so a preflight
decides whether the mutation happens. A preflight that stops answering is
invisible to curl and to every server side probe, and one already took a form
down that way without leaving a trace anywhere.

The origin asked about is `SITE_ORIGIN`, the production domain, even when
`SITE_URL` points at a preview deployment: what is under test is whether a
visitor on limosen.at may make the call.

In [ ]:
ORIGIN = k.CONFIG["site_origin"]

with k.check("the preflight from this origin is answered") as c:
    r = c.require(k.preflight(k.CONFIG["emailwerk_url"], ORIGIN))
    c.expect(r.status in (200, 204), f"OPTIONS answers 200 or 204 (got {r.status})")
    allow_origin = r.header("access-control-allow-origin")
    c.expect(allow_origin == ORIGIN, f"allow-origin is {ORIGIN} (got {allow_origin!r})")
    c.expect("POST" in r.header("access-control-allow-methods").upper(),
             f"POST is allowed (got {r.header('access-control-allow-methods')!r})")
    c.expect("content-type" in r.header("access-control-allow-headers").lower(),
             f"content-type may be sent (got {r.header('access-control-allow-headers')!r})")

## 5. The anonymous path is open, and the shape is the deployed one

This is the one probe that reaches the send resolver, and it is free of side
effects by construction. emailwerk validates the arguments **before** the rate
limit and **before** it looks the template up, so a reply-to that is not an
address is refused as `PUBLIC_SEND_INVALID_REPLY_TO` with nothing created, no
message row, no mail, no counter moved. What it does prove is the entire path
the form uses: CORS, the anonymous gate, the deployed schema and the resolver
behind it.

The three other answers are named explicitly, because each of them means
something different and one of them has already happened here:
`GRAPHQL_VALIDATION_FAILED` would say the document the site sends no longer
matches the deployed schema, HTTP 401 would say the anonymous gate is shut, and
`TEMPLATE_NOT_FOUND` would say the deployed id is gone or is no longer public.

That last one is named to be ruled out, not to be found: emailwerk validates the
arguments before it looks the template up, so the reply-to is refused first and
this probe can never reach a missing or private template. Asserting the code is
absent says the answer is the argument refusal and not something else; it says
nothing about whether the id still exists. Section 6's admin read is the only
thing that proves that.

The probe is sent **without** the admin credential even when one is in the
environment. A request that carries `Authorization` takes emailwerk's
authenticated branch, which validates other things in another order, and then
this probe proves nothing about the path a visitor walks.

In [ ]:
SEND = ("mutation($args: SendTemplateMailArgsInput!)"
        "{ sendTemplateMail(args:$args){ id status } }")

with k.check("an invalid reply-to is refused, which proves the path without sending") as c:
    probe_id = DEPLOYED.get("de")
    if not probe_id:
        c.fail("no deployed de id to probe with")
    else:
        r = c.require(k.emailwerk(SEND, {"args": {
            "templateId": probe_id,
            "envelopeOverride": {"replyTo": "not-an-address"},
            "values": {"firstName": "post-deploy", "message": "argument probe, nothing is sent"},
        }}, auth="", origin=ORIGIN))
        c.expect(r.status == 200, f"HTTP 200 with the error in the body (got {r.status})")
        c.expect(len(r.errors) == 1, f"exactly one error (got {len(r.errors)}: {r.codes})")
        c.expect("PUBLIC_SEND_INVALID_REPLY_TO" in r.codes,
                 f"refused as an invalid reply-to (got {r.codes})")
        c.expect("GRAPHQL_VALIDATION_FAILED" not in r.codes,
                 "the mutation still has the shape the site sends")
        c.expect(r.status != 401 and "AUTH_REQUIRED" not in r.codes,
                 "a signed-out visitor is not turned away at the gate")
        c.expect("TEMPLATE_NOT_FOUND" not in r.codes,
                 "the refusal is the reply-to one, not a missing template")

## 6. The templates the deployed site sends

The admin read, and the check that would have caught 2026-09-07. For each of
the four ids the bundle actually carries: it is readable, it is public (a
signed-out visitor is refused a private template before a message row exists,
which leaves no trace at all), it does not demand a verified reply-to or a
signature (either makes a template not publicly sendable), it sends from this
tenant's verified sender, it has a subject, its description belongs to this
brand, and **its stored envelope names this company's recipients**. The last
one is the whole incident: the recipients are what the anonymous branch
delivers to, and an empty list is a refused send.

Without `EMAILWERK_AUTH` this half records SKIP, so the notebook still runs and
still says something useful. On a runner it records FAIL instead: the gate
exists for this read.

In [ ]:
TEMPLATE = """
query($a: TemplateArgsInput!) {
  template(args: $a) {
    id description engine isPublic verifyReplyTo requiresSignature
    senderId parentId updatedAt
    envelope { subject to replyTo }
    variables { name isRequired }
    content
  }
}
"""

TEMPLATES = {}

with k.check("every deployed template is readable, public and sendable anonymously") as c:
    if c.missing_auth():
        pass
    elif not DEPLOYED:
        c.fail("no deployed ids to read")
    else:
        for lang in LANGUAGES:
            template_id = DEPLOYED.get(lang)
            if not template_id:
                continue
            r = k.emailwerk(TEMPLATE, {"a": {"id": template_id}})
            node = (r.data or {}).get("template")
            if not node:
                c.fail(f"{lang} {template_id} is not readable: {r.error_message or r.status}")
                continue
            TEMPLATES[lang] = node
            c.expect(node.get("isPublic") is True, f"{lang} is public")
            c.expect(node.get("verifyReplyTo") is False, f"{lang} does not demand a verified reply-to")
            c.expect(node.get("requiresSignature") is False, f"{lang} does not demand a signature")
            c.expect(node.get("senderId") == SENDER_ID,
                     f"{lang} sends from {SENDER_ID} (got {node.get('senderId')})")
            c.expect((node.get("description") or "").startswith(k.DESCRIPTION_PREFIX),
                     f"{lang} is described as {k.DESCRIPTION_PREFIX!r}… (got {node.get('description')!r})")

with k.check("every deployed template still carries this company's recipients") as c:
    if not TEMPLATES:
        c.skip("no templates were read")
    else:
        for lang in LANGUAGES:
            node = TEMPLATES.get(lang)
            if not node:
                continue
            envelope = node.get("envelope") or {}
            to = list(envelope.get("to") or [])
            c.expect(bool(to), f"{lang} has a recipient at all")
            c.expect(set(to) == RECIPIENTS,
                     f"{lang} goes to {sorted(RECIPIENTS)} (got {sorted(to)})")
            c.expect(bool((envelope.get("subject") or "").strip()), f"{lang} has a subject")
            c.note(f"{lang}: to={sorted(to)} updatedAt={node.get('updatedAt')}")
        c.note("an empty list here is the 2026-09-07 failure: the anonymous branch "
               "delivers to the stored envelope and refuses a template without one")

## 7. Brand purity of the templates, and the palette

The same question as section 3, asked of what emailwerk stores: no recipient,
no subject and no body of a template this site sends may name the other
company. The bodies of the confirmation children are read too, because the
server sends those to the visitor and nobody here ever sees them.

The palette is measured, not assumed. limosen's gold is `#d4af37` with
`#b8932f` beside it on charcoal `#1b1b1b`, in `src/styles/theme/system.ts` and
in the templates alike. `#bf9c60` is KRC's muted gold from krclimo.at, so a
limosen mail that carries it is the brand mix in a place the deploy script
cannot look. A missing `#d4af37` is only a warning: a template may be redesigned
without the gold, and that is a decision, not a defect.

In [ ]:
CHILD_QUERY = TEMPLATE


def palette(c, lang, what, body):
    """This brand's gold in a mail, and the sibling's.

    One function for both the request a visitor triggers and the confirmation
    the server sends back, because the question is the same and only the sibling
    repository's answer differs: there the same finding is an open item and a
    warning, here it is a failure.
    """
    text = (body or "").lower()
    c.expect(k.FOREIGN_GOLD not in text,
             f"{lang} {what} is free of KRC's {k.FOREIGN_GOLD}")
    if k.BRAND_GOLD not in text:
        c.warn(f"{lang} {what} no longer carries limosen's {k.BRAND_GOLD}")


with k.check("no template of this site names the other company") as c:
    if not TEMPLATES:
        c.skip("no templates were read")
    else:
        for lang, node in sorted(TEMPLATES.items()):
            envelope = node.get("envelope") or {}
            where = {
                "recipients": " ".join(envelope.get("to") or []),
                "subject": envelope.get("subject") or "",
                "body": node.get("content") or "",
            }
            for part, text in where.items():
                found = [m for m in k.foreign_markers() if m in text]
                c.expect(not found, f"{lang} request {part} is clean (found {found})")

with k.check("the mail a visitor reads carries this brand's palette") as c:
    if not TEMPLATES:
        c.skip("no templates were read")
    else:
        for lang, node in sorted(TEMPLATES.items()):
            palette(c, lang, "request", node.get("content"))

## 8. The confirmation answers in the visitor's language

The site sends the parent only. The server enqueues the parent's public child
to `envelope.replyTo` by itself, so the link is what decides which language the
visitor is answered in, and a second send from the site would double the mail.

Two properties matter and both have been wrong here before: exactly one public
child per request, so there is no ambiguity about which one the server picks,
and an empty `to` on that child, because a recipient there would send every
visitor's confirmation to one fixed address.
The tenant is read once here, and the page is asserted to be whole before
anything is counted: a child that is merely past the end of the page is
indistinguishable from a child that was never created, and reporting a healthy
tenant as a broken one is how a good deployment gets rolled back. The
confirmation bodies are kept as they are read, because the palette check that
closes this section needs them and asking emailwerk for them twice would be
asking the same question twice.

In [ ]:
LIST = ("""
query($a: TemplateListArgsInput) {
  templates(args: $a) {
    nodes { id description isPublic parentId envelope { subject to } }
    totalCount
  }
}
""")

# The tenant is read once, here, and both checks below work off that one page.
# `first` is asked for generously and the count is then asserted rather than
# trusted, because a child that fell off the end of a page looks exactly like a
# child that was never created: section 8 would report `de has exactly one
# public child (got 0)` on a perfectly healthy tenant, and under the workflow
# that reads as a broken deployment and rolls a good one back. 122 rows live
# here today and 1000 was measured as honoured, so the margin is large; what
# matters is that the day it stops being large, the run says so.
LISTED = {"call": None, "nodes": [], "total": 0}
if k.have_emailwerk_auth():
    LISTED["call"] = k.emailwerk(LIST, {"a": {"first": 1000}})
    listed = ((LISTED["call"].data or {}).get("templates") or {})
    LISTED["nodes"] = listed.get("nodes") or []
    LISTED["total"] = listed.get("totalCount") or 0

# The confirmation bodies, kept as they are read, so the palette check below
# does not have to fetch them a second time.
CONFIRMATIONS = {}

with k.check("each deployed request has exactly one public child, and it has no recipient") as c:
    if c.missing_auth():
        pass
    elif not DEPLOYED:
        c.fail("no deployed ids to look for children of")
    else:
        c.require(LISTED["call"])
        nodes = LISTED["nodes"]
        c.expect(bool(nodes), f"the template list is readable ({len(nodes)} rows)")
        c.expect(LISTED["total"] <= len(nodes),
                 f"the whole tenant is on one page ({len(nodes)} of {LISTED['total']} rows)")
        for lang in LANGUAGES:
            parent_id = DEPLOYED.get(lang)
            if not parent_id:
                continue
            children = [n for n in nodes if n.get("parentId") == parent_id and n.get("isPublic")]
            c.expect(len(children) == 1,
                     f"{lang} has exactly one public child (got {len(children)})")
            for child in children:
                c.expect(not (child.get("envelope") or {}).get("to"),
                         f"{lang} confirmation has an empty to, the server answers replyTo")
                c.expect((child.get("description") or "").startswith(k.DESCRIPTION_PREFIX),
                         f"{lang} confirmation belongs to this brand "
                         f"({child.get('description')!r})")
                c.note(f"{lang}: {child.get('id')} {child.get('description')!r}")

with k.check("no confirmation body names the other company either") as c:
    if not (TEMPLATES and k.have_emailwerk_auth()):
        c.skip("no templates were read")
    else:
        for lang in LANGUAGES:
            parent_id = DEPLOYED.get(lang)
            child = next((n for n in LISTED["nodes"]
                          if n.get("parentId") == parent_id and n.get("isPublic")), None)
            if not child:
                continue
            body_result = k.emailwerk(CHILD_QUERY, {"a": {"id": child["id"]}})
            body_node = (body_result.data or {}).get("template") or {}
            body = body_node.get("content") or ""
            CONFIRMATIONS[lang] = body
            found = [m for m in k.foreign_markers() if m in body]
            c.expect(not found, f"{lang} confirmation body is clean (found {found})")

with k.check("the confirmation a visitor reads carries this brand's palette") as c:
    if not CONFIRMATIONS:
        c.skip("no confirmation bodies were read")
    else:
        for lang, body in sorted(CONFIRMATIONS.items()):
            palette(c, lang, "confirmation", body)


## 9. The source the deployed bundle was built from

The two checks above read the live site. This one reads the checkout, so that a
template run which wrote new ids but was never deployed, or a source file that
was edited past its own ids.json, is named here rather than showing up as a
mysterious drift warning in section 2.

In [ ]:
with k.check("mail-templates.ts maps every language to its request id in ids.json") as c:
    root = k.repo_root()
    if root is None:
        c.skip("not inside the checkout")
    else:
        source = (root / "src" / "services" / "mail-templates.ts").read_text(encoding="utf-8")
        for lang in LANGUAGES:
            expected = REPO_REQUEST.get(lang)
            c.expect(bool(expected), f"ids.json has {k.TEMPLATE_PREFIX}{lang}-request")
            if not expected:
                continue
            found = re.search(rf"\b{lang}:\s*'([^']+)'", source)
            c.expect(bool(found), f"mail-templates.ts names {lang}")
            if found:
                c.expect(found.group(1) == expected,
                         f"{lang} is {expected} in both (source says {found.group(1)})")

## 9b. The booking form's backend is reachable from the browser

A booking is two calls, not one: the form posts `bookTransfer` to this brand's
own pylon first and only sends the enquiry mail afterwards, with the code the
pylon minted. So the preflight to the pylon decides whether a booking is
created at all, and a booking that is refused looks almost like success from
the outside, an orange "Booking not yet in the system" followed by a green one
for the mail.

The URL is read out of the deployed bundle rather than out of the checkout, and
then compared with what the checkout would build, so a site built against the
wrong backend is visible here.

In [ ]:
BUNDLE_PYLONS = k.pylon_urls_in_bundle(SITE["bundle"])
DEPLOYED_PYLON = BUNDLE_PYLONS[0] if BUNDLE_PYLONS else ""

with k.check("the deployed bundle talks to this brand's own pylon") as c:
    c.expect(bool(DEPLOYED_PYLON), f"the bundle names a pylon (found {BUNDLE_PYLONS})")
    c.expect(DEPLOYED_PYLON == k.PYLON_URL,
             f"it is {k.PYLON_URL} (got {DEPLOYED_PYLON!r})")
    in_repo = k.repo_pylon_url()
    if in_repo:
        c.expect(in_repo == DEPLOYED_PYLON,
                 f"site-variants.ts agrees ({in_repo})")

with k.check("the preflight to the pylon is answered for this origin") as c:
    target = DEPLOYED_PYLON or k.PYLON_URL
    r = c.require(k.preflight(target, ORIGIN))
    c.expect(r.status in (200, 204), f"OPTIONS answers 200 or 204 (got {r.status})")
    c.expect(r.header("access-control-allow-origin") == ORIGIN,
             f"allow-origin is {ORIGIN} (got {r.header('access-control-allow-origin')!r})")
    c.expect("POST" in r.header("access-control-allow-methods").upper(),
             f"POST is allowed (got {r.header('access-control-allow-methods')!r})")

## 9c. The booking mutation is still the one the site posts

Proven without writing a row. The document carries one field that does not
exist, so the call is refused during schema validation, before a resolver runs
and before anything reaches the database. What survives that refusal is the
interesting part: an error that names the planted field means `bookTransfer`
is deployed and takes an `args` object, which is exactly the contract the site
builds its literal document against. The complaints about the required fields
that were left out come along with it and are expected.

This mirrors what `~/git/taxi-app/tests/09-website-booking.ipynb` does on the
API side, for the same reason: the site once posted the flat argument list of a
schema that had taken a single `args` object for months, and every booking made
here was refused.

In [ ]:
PLANTED = "thisFieldDoesNotExist_selftest"

with k.check("bookTransfer is deployed, takes args, and stops at validation") as c:
    target = DEPLOYED_PYLON or k.PYLON_URL
    r = c.require(k.gql(target,
                        "mutation { bookTransfer(args: {%s: 1}) { id } }" % PLANTED,
                        headers={"Origin": ORIGIN}))
    messages = [e.get("message", "") for e in r.errors]
    c.expect(not r.ok, "the probe is refused, as designed")
    c.expect(any(PLANTED in m for m in messages),
             f"an error names the field we planted (got {messages[:4]})")
    c.expect(all((e.get("extensions") or {}).get("code") == "GRAPHQL_VALIDATION_FAILED"
                 for e in r.errors),
             "every error is a validation error, so nothing was written")
    c.expect(not any("Cannot query field" in m and "bookTransfer" in m for m in messages),
             "bookTransfer itself is still a field of Mutation")
    c.expect(not any('Unknown argument "args"' in m for m in messages),
             "it still takes an argument called args")

## 9d. The pylon is this brand's own

The sibling site's backend answers the same schema, so a bundle built against
`api.booklimo.at` would work, and every booking made on limosen.at would land
in KRC's dispatch. Nothing else in the stack would notice.

In [ ]:
with k.check("no sibling pylon host is in the bundle") as c:
    others = [url for url in BUNDLE_PYLONS if url != k.PYLON_URL]
    c.expect(not others, f"the bundle names only {k.PYLON_URL} (also found {others})")
    c.expect("api.booklimo.at" not in SITE["bundle"],
             "the sibling's API host does not appear anywhere in the bundle")

## 10. The drill

`SMOKE_DRILL=1` fails this one check on purpose and changes nothing else. The
workflow sets it for the first production run when a drill was asked for, so
the plan, the previous-deployment check, a real rollback and the recheck after
it can be exercised on a healthy system rather than the first time they are
needed. See `tests/README.md`.

In [ ]:
with k.check("the drill") as c:
    if k.CONFIG["drill"]:
        c.fail("drill: forced failure requested by SMOKE_DRILL")
    else:
        c.skip("SMOKE_DRILL is not set")

## Sending, for a human to run

Nothing above sends. If a real end to end send is wanted, for instance after a
template change, this is the recipe. **It puts mail in the company's inbox**:
the request template's stored recipients are `office@limosen.at` and
`limosen@netsnek.com`, and the server additionally answers the reply-to with
the German confirmation. Use this brand's netsnek copy address as the reply-to,
never a customer's, and never the sibling brand's.

One anonymous German enquiry, exactly as the form sends it:

```sh
curl -s https://emailwerk.com/graphql \
  -H 'content-type: application/json' \
  -H 'origin: https://limosen.at' \
  -d '{"query":"mutation($args: SendTemplateMailArgsInput!){ sendTemplateMail(args:$args){ id status } }",
       "variables":{"args":{
         "templateId":"<the de id printed in section 2>",
         "envelopeOverride":{"replyTo":"limosen@netsnek.com"},
         "values":{"firstName":"Post","lastName":"Deploy","email":"limosen@netsnek.com",
                   "phone":"+43 660 000 0000","message":"Testanfrage nach dem Deploy.",
                   "invokedOnUrl":"https://limosen.at/"}}}}'
```

It answers `{"data":{"sendTemplateMail":{"id":"<message id>","status":"QUEUED"}}}`.
Read what became of that row with the admin credential, which is
`~/.config/taxi-app/emailwerk-basic.txt` locally and the repository secret
`EMAILWERK_AUTH` on a runner:

```sh
EMAILWERK_AUTH="$(cat ~/.config/taxi-app/emailwerk-basic.txt)" \
python3 - <<'PY'
import base64, json, os, urllib.request
query = "query($a: MessageArgsInput!){ message(args:$a){ id status error toAddress subject } }"
body = json.dumps({"query": query, "variables": {"a": {"id": "<message id>"}}}).encode()
request = urllib.request.Request("https://emailwerk.com/graphql", data=body, headers={
    "Content-Type": "application/json",
    "Authorization": "Basic " + base64.b64encode(os.environ["EMAILWERK_AUTH"].encode()).decode()})
print(urllib.request.urlopen(request, timeout=30).read().decode())
PY
```

`status` walks `QUEUED` to `SENT`; anything else carries the reason in `error`.
Do not paste the credential into a shell where it lands in the history, and
never into a notebook cell.

In [ ]:
k.finish()